[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/08_Model_Optimization_and_Quantization/02_Quantization_Techniques/Quantization_Techniques_Deep_Dive.ipynb)

# 8.2 Quantization Techniques — Deep Dive

## Table of Contents
1. [Quantization Fundamentals](#section-1)
2. [Manual Quantize / Dequantize Implementation](#section-2)
3. [Error Analysis](#section-3)
4. [Symmetric vs Asymmetric Quantization](#section-4)
5. [Per-Tensor vs Per-Channel Quantization](#section-5)
6. [Static vs Dynamic Quantization](#section-6)
7. [Post-Training Quantization (PTQ)](#section-7)
8. [Quantization-Aware Training (QAT) and STE](#section-8)
9. [Calibration Methods](#section-9)
10. [ONNX Quantization Toolchain and QDQ Format](#section-10)
11. [Dynamic Quantization with ONNX Runtime](#section-11)
12. [Static Quantization with ONNX Runtime](#section-12)
13. [Model Size and Performance Comparison](#section-13)
14. [FP32 vs FP16 vs INT8 Comparison](#section-14)
15. [Summary](#section-15)

![Quantization Concepts](assets/quantization_concepts.png)

<a id='section-1'></a>
## Section 1: Quantization Fundamentals

### The Core Idea

Quantization maps values from a **continuous (or high-precision) domain** to a **discrete (lower-precision) domain**. In deep learning, this typically means converting FP32 tensors to INT8, reducing storage from 32 bits per element to 8 bits — a **4× compression**.

```
 FP32 number line (continuous, ~2^24 mantissa levels of precision):
 ───────┼──────────────────────────────────┼────────────────────────
      r_min                              r_max
        ╎  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ╎     (billions of representable values)

                    │  QUANTIZE  │
                    ▼            ▼

 INT8 lattice (discrete, 256 levels):
 ───────┼────┼────┼────┼────┼────┼────┼────┼────────────────────────
       -128 -96  -64  -32    0   32   64  96  127
        q_min ◄──── 256 bins ────► q_max
```

### Formal Definitions

**Affine (linear) quantization** maps a real value $r$ to an integer $q$ via:

#### Quantize

$$Q(r) = \text{clamp}\left(\left\lfloor \frac{r}{s} \right\rceil + z,\; q_{\min},\; q_{\max}\right)$$

where $\lfloor \cdot \rceil$ denotes *round-to-nearest-even* (banker's rounding).

#### Dequantize

$$D(q) = s \cdot (q - z)$$

This recovers an *approximation* of the original real value.

#### Scale

The scale $s$ maps the real-valued range $[r_{\min}, r_{\max}]$ onto the integer range $[q_{\min}, q_{\max}]$:

$$s = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}$$

For INT8 unsigned: $q_{\min}=0, q_{\max}=255$. For INT8 signed: $q_{\min}=-128, q_{\max}=127$.

#### Zero-Point

The zero-point $z$ ensures that real zero maps *exactly* to an integer value (critical for padding and ReLU):

$$z = q_{\min} - \left\lfloor \frac{r_{\min}}{s} \right\rceil$$

### Derivation of Scale and Zero-Point

We require two boundary conditions:

$$D(q_{\min}) = r_{\min} \implies s \cdot (q_{\min} - z) = r_{\min}$$
$$D(q_{\max}) = r_{\max} \implies s \cdot (q_{\max} - z) = r_{\max}$$

Subtracting:

$$s \cdot (q_{\max} - q_{\min}) = r_{\max} - r_{\min}$$

$$\boxed{s = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}}$$

From the first equation:

$$z = q_{\min} - \frac{r_{\min}}{s}$$

Since $z$ must be an integer, we round:

$$\boxed{z = q_{\min} - \left\lfloor \frac{r_{\min}}{s} \right\rceil}$$

### The Quantize-Dequantize Round-Trip

```
 r (FP32)  ──► Q(r) = clamp(⌊r/s⌉ + z, q_min, q_max)  ──► q (INT8)
                                                               │
                                                               │  store / transmit
                                                               │  as 8-bit integer
                                                               ▼
 r̂ (FP32)  ◄── D(q) = s · (q - z)                       ◄── q (INT8)

 Reconstruction error:  |r - r̂| ≤ s/2  (for values in range)
```

In [ ]:
!pip install onnx onnxruntime numpy matplotlib -q

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper
from onnx.checker import check_model
import matplotlib.pyplot as plt
import os, tempfile, struct

print(f"ONNX version: {onnx.__version__}")

try:
    import onnxruntime as ort
    print(f"onnxruntime version: {ort.__version__}")
except ImportError:
    print("onnxruntime not available")

try:
    from onnxruntime.quantization import quantize_dynamic, quantize_static, QuantType, CalibrationMethod
    print("onnxruntime.quantization available")
except ImportError:
    print("onnxruntime.quantization not available")

<a id='section-2'></a>
## Section 2: Manual Quantize / Dequantize Implementation

Before using any library, let's implement the core operations from scratch to internalize the math.

In [ ]:
def compute_scale_zp(r_min, r_max, q_min, q_max):
    """Derive scale and zero-point from real and quantized ranges."""
    s = (r_max - r_min) / (q_max - q_min)
    z = int(np.round(q_min - r_min / s))
    z = max(q_min, min(q_max, z))
    return s, z


def quantize(r, s, z, q_min, q_max):
    """Q(r) = clamp(round(r / s) + z, q_min, q_max)"""
    return np.clip(np.round(r / s).astype(np.int32) + z, q_min, q_max).astype(np.int8)


def dequantize(q, s, z):
    """D(q) = s * (q - z)"""
    return s * (q.astype(np.float32) - z)


np.random.seed(42)
r = np.random.randn(1000).astype(np.float32)

r_min, r_max = float(r.min()), float(r.max())
q_min, q_max = -128, 127  # signed INT8

s, z = compute_scale_zp(r_min, r_max, q_min, q_max)

q = quantize(r, s, z, q_min, q_max)
r_hat = dequantize(q, s, z)

print("Manual Quantize / Dequantize")
print("=" * 50)
print(f"Real range:       [{r_min:.4f}, {r_max:.4f}]")
print(f"Quantized range:  [{q_min}, {q_max}]")
print(f"Scale (s):        {s:.6f}")
print(f"Zero-point (z):   {z}")
print(f"")
print(f"Sample values (first 8):")
print(f"  Original r:     {r[:8].round(4)}")
print(f"  Quantized q:    {q[:8]}")
print(f"  Recovered r̂:    {r_hat[:8].round(4)}")
print(f"  Error |r-r̂|:    {np.abs(r[:8] - r_hat[:8]).round(6)}")
print(f"")
print(f"Max |r - r̂|:      {np.max(np.abs(r - r_hat)):.6f}")
print(f"Theoretical bound (s/2): {s/2:.6f}")
print(f"Bound holds: {np.max(np.abs(r - r_hat)) <= s/2 + 1e-7}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Original vs reconstructed values
idx = np.argsort(r)[:200]
axes[0].plot(r[idx], label='Original r', color='#2196F3', linewidth=1.5)
axes[0].plot(r_hat[idx], label='Recovered r̂', color='#FF5722', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Sample index (sorted)', fontsize=11)
axes[0].set_ylabel('Value', fontsize=11)
axes[0].set_title('Original vs Quantized-Dequantized', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Plot 2: Quantization error distribution
errors = r - r_hat
axes[1].hist(errors, bins=50, color='#9C27B0', edgecolor='black', alpha=0.8)
axes[1].axvline(s/2, color='red', linestyle='--', linewidth=2, label=f'+s/2 = {s/2:.4f}')
axes[1].axvline(-s/2, color='red', linestyle='--', linewidth=2, label=f'-s/2 = {-s/2:.4f}')
axes[1].set_xlabel('Quantization Error (r - r̂)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Error Distribution', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Plot 3: Quantization staircase
r_sorted = np.linspace(r_min, r_max, 500).astype(np.float32)
q_sorted = quantize(r_sorted, s, z, q_min, q_max)
r_hat_sorted = dequantize(q_sorted, s, z)
axes[2].plot(r_sorted, r_sorted, 'b-', label='Identity (no quantization)', linewidth=1.5)
axes[2].plot(r_sorted, r_hat_sorted, 'r-', label='Quantize → Dequantize', linewidth=1.5)
axes[2].set_xlabel('Input r', fontsize=11)
axes[2].set_ylabel('Output r̂', fontsize=11)
axes[2].set_title('Quantization Staircase Function', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-3'></a>
## Section 3: Error Analysis

### Quantization Error Bound

For a value $r$ within $[r_{\min}, r_{\max}]$, the quantize-dequantize round-trip introduces error:

$$|r - D(Q(r))| \leq \frac{s}{2}$$

**Proof:** The nearest integer $q = \lfloor r/s \rceil + z$ satisfies $|r/s - (q - z)| \leq 1/2$. Multiplying by $s$: $|r - s(q-z)| = |r - D(q)| \leq s/2$. $\square$

### MSE for Uniform Distribution

If $r$ is uniformly distributed within each quantization bin of width $s$, the quantization error $e = r - D(Q(r))$ is uniformly distributed on $[-s/2, s/2]$:

$$\text{MSE} = \mathbb{E}[e^2] = \frac{1}{s}\int_{-s/2}^{s/2} e^2 \, de = \frac{1}{s} \cdot \frac{s^3}{12} = \frac{s^2}{12}$$

$$\boxed{\text{MSE}_{\text{uniform}} = \frac{s^2}{12}}$$

### Signal-to-Quantization-Noise Ratio (SQNR)

The SQNR measures how much the signal power exceeds the quantization noise:

$$\text{SQNR} = 10 \log_{10} \frac{\sigma_r^2}{\text{MSE}} = 10 \log_{10} \frac{\sigma_r^2}{s^2 / 12}$$

For $n$-bit quantization with full-range signal ($\sigma_r^2 \approx (2^n s)^2 / 12$):

$$\text{SQNR} \approx 6.02n + 1.76 \;\text{dB}$$

| Bits | SQNR (dB) | Interpretation |
|:----:|:---------:|:---:|
| 8 | ~49.9 | Good for most inference |
| 4 | ~25.8 | Aggressive, accuracy-sensitive |
| 2 | ~13.8 | Research frontier |
| 16 | ~98.1 | Near-lossless for weights |

### Accumulated Error Through Layers

For a network with $L$ sequential quantized linear layers (each with independent quantization error), the output error grows approximately as:

$$\text{MSE}_{\text{output}} \approx \sum_{l=1}^{L} \left(\prod_{j=l+1}^{L} \|W_j\|_F^2 \right) \cdot \text{MSE}_l$$

Layers with large weight norms **amplify** quantization error from earlier layers. This is why:
- First/last layers are often kept at higher precision
- Per-channel quantization helps layers with high dynamic range
- Calibration data quality matters most for high-norm layers

In [ ]:
def compute_sqnr(signal, quantized_signal):
    """Compute Signal-to-Quantization-Noise Ratio in dB."""
    noise = signal - quantized_signal
    signal_power = np.mean(signal ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power == 0:
        return float('inf')
    return 10 * np.log10(signal_power / noise_power)


np.random.seed(42)
signal = np.random.randn(10000).astype(np.float32)

bit_widths = [2, 3, 4, 5, 6, 7, 8, 10, 12, 16]
sqnr_values = []
mse_values = []

for bits in bit_widths:
    n_levels = 2 ** bits
    qmin, qmax = -(n_levels // 2), (n_levels // 2) - 1
    s_b, z_b = compute_scale_zp(float(signal.min()), float(signal.max()), qmin, qmax)
    q_b = np.clip(np.round(signal / s_b).astype(np.int32) + z_b, qmin, qmax)
    r_hat_b = s_b * (q_b.astype(np.float32) - z_b)
    sqnr_values.append(compute_sqnr(signal, r_hat_b))
    mse_values.append(np.mean((signal - r_hat_b) ** 2))

theoretical_sqnr = [6.02 * b + 1.76 for b in bit_widths]

print("Error Analysis Across Bit Widths")
print("=" * 60)
print(f"{'Bits':>5} {'SQNR (meas.)':>14} {'SQNR (theory)':>15} {'MSE':>12} {'s²/12':>12}")
print("-" * 60)
for i, bits in enumerate(bit_widths):
    n_levels = 2 ** bits
    qmin_b, qmax_b = -(n_levels // 2), (n_levels // 2) - 1
    s_b = (signal.max() - signal.min()) / (qmax_b - qmin_b)
    theoretical_mse = s_b**2 / 12
    print(f"{bits:>5d} {sqnr_values[i]:>12.2f} dB {theoretical_sqnr[i]:>12.2f} dB {mse_values[i]:>12.6f} {theoretical_mse:>12.6f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: SQNR vs bit width
axes[0].plot(bit_widths, sqnr_values, 'bo-', linewidth=2, markersize=8, label='Measured')
axes[0].plot(bit_widths, theoretical_sqnr, 'r--', linewidth=2, label='Theory: 6.02n + 1.76')
axes[0].set_xlabel('Bit Width', fontsize=12)
axes[0].set_ylabel('SQNR (dB)', fontsize=12)
axes[0].set_title('SQNR vs Quantization Bit Width', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].set_xticks(bit_widths)

# Plot 2: MSE vs bit width (log scale)
axes[1].semilogy(bit_widths, mse_values, 'gs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Bit Width', fontsize=12)
axes[1].set_ylabel('MSE (log scale)', fontsize=12)
axes[1].set_title('Quantization MSE vs Bit Width', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].set_xticks(bit_widths)

# Plot 3: Error accumulation through layers
np.random.seed(0)
n_layers_range = range(1, 21)
accumulated_mse_8bit = []
accumulated_mse_16bit = []

for n_layers in n_layers_range:
    x = np.random.randn(100).astype(np.float32)
    x_q8 = x.copy()
    x_q16 = x.copy()
    for _ in range(n_layers):
        W = np.random.randn(100, 100).astype(np.float32) * 0.1
        x = x @ W
        x_q8 = x_q8 @ W
        x_q16 = x_q16 @ W
        # quantize activations
        for arr, bits in [(x_q8, 8), (x_q16, 16)]:
            n_levels = 2 ** bits
            qmin_l, qmax_l = -(n_levels // 2), (n_levels // 2) - 1
            if arr.max() - arr.min() > 1e-10:
                s_l, z_l = compute_scale_zp(float(arr.min()), float(arr.max()), qmin_l, qmax_l)
                q_l = np.clip(np.round(arr / s_l).astype(np.int32) + z_l, qmin_l, qmax_l)
                arr[:] = s_l * (q_l.astype(np.float32) - z_l)
    accumulated_mse_8bit.append(np.mean((x - x_q8) ** 2))
    accumulated_mse_16bit.append(np.mean((x - x_q16) ** 2))

axes[2].semilogy(list(n_layers_range), accumulated_mse_8bit, 'r^-', linewidth=2, label='INT8')
axes[2].semilogy(list(n_layers_range), accumulated_mse_16bit, 'b^-', linewidth=2, label='INT16')
axes[2].set_xlabel('Number of Layers', fontsize=12)
axes[2].set_ylabel('Accumulated MSE (log)', fontsize=12)
axes[2].set_title('Error Accumulation Through Layers', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-4'></a>
## Section 4: Symmetric vs Asymmetric Quantization

### Symmetric Quantization

In symmetric quantization, the zero-point is forced to 0 ($z = 0$), and the scale is determined by the maximum absolute value:

$$s_{\text{sym}} = \frac{2 \cdot \max(|r|)}{q_{\max} - q_{\min}} = \frac{\max(|r|)}{127} \quad \text{(for signed INT8)}$$

The mapping becomes:

$$Q_{\text{sym}}(r) = \text{clamp}\left(\left\lfloor \frac{r}{s_{\text{sym}}} \right\rceil, -128, 127\right)$$
$$D_{\text{sym}}(q) = s_{\text{sym}} \cdot q$$

**Advantage:** Simpler arithmetic (no zero-point subtraction), real zero maps exactly to integer zero.

**Disadvantage:** Wastes range when data is asymmetric (e.g., post-ReLU activations are all non-negative).

### Asymmetric Quantization

Asymmetric quantization uses the full $[q_{\min}, q_{\max}]$ range with $z \neq 0$:

$$s_{\text{asym}} = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}, \quad z = q_{\min} - \left\lfloor \frac{r_{\min}}{s_{\text{asym}}} \right\rceil$$

**Advantage:** Full utilization of the integer range — better accuracy for skewed distributions.

**Disadvantage:** Extra zero-point arithmetic in the inner loop.

### Visual Comparison

```
 Symmetric (z=0):  real range is [-max|r|, +max|r|]
 ──────────┼──────────┼──────────┼──────────┼──────────
         -max|r|       0         +max|r|
           ↕           ↕           ↕
         -128          0          127
           ▲                       ▲
       wasted if r_min > -max|r|   fully used

 Asymmetric (z≠0): real range is [r_min, r_max]
 ──────────┼──────────┼──────────┼──────────
         r_min        real 0     r_max
           ↕           ↕           ↕
         q_min     z (int zero)  q_max
           ▲                       ▲
       fully used                fully used

 Example: post-ReLU data r ∈ [0, 6.0]
   Symmetric:  s = 6.0/127 = 0.0472  (q ∈ [0, 127], wastes [-128, -1])
   Asymmetric: s = 6.0/255 = 0.0235  (q ∈ [0, 255], uses all 256 levels)
                     → 2× better resolution!
```

In [ ]:
def symmetric_quantize(r, bits=8):
    """Symmetric quantization: z=0, scale based on max(|r|)."""
    q_max = (2 ** (bits - 1)) - 1
    q_min = -(2 ** (bits - 1))
    s = np.max(np.abs(r)) / q_max
    q = np.clip(np.round(r / s), q_min, q_max).astype(np.int8)
    r_hat = s * q.astype(np.float32)
    return q, r_hat, s, 0


def asymmetric_quantize(r, bits=8):
    """Asymmetric quantization: full range utilization."""
    q_max = (2 ** bits) - 1
    q_min = 0
    s = (r.max() - r.min()) / (q_max - q_min)
    z = int(np.round(q_min - r.min() / s))
    z = max(q_min, min(q_max, z))
    q = np.clip(np.round(r / s).astype(np.int32) + z, q_min, q_max).astype(np.uint8)
    r_hat = s * (q.astype(np.float32) - z)
    return q, r_hat, s, z


np.random.seed(42)

# Case 1: Symmetric data (weights, mean ≈ 0)
weights = np.random.randn(5000).astype(np.float32) * 0.5

# Case 2: Asymmetric data (post-ReLU activations, all ≥ 0)
activations = np.maximum(0, np.random.randn(5000).astype(np.float32) * 2.0)

print("Symmetric vs Asymmetric Quantization")
print("=" * 60)

for name, data in [("Weights (symmetric data)", weights), ("ReLU activations (asymmetric data)", activations)]:
    _, r_hat_sym, s_sym, z_sym = symmetric_quantize(data)
    _, r_hat_asym, s_asym, z_asym = asymmetric_quantize(data)
    mse_sym = np.mean((data - r_hat_sym) ** 2)
    mse_asym = np.mean((data - r_hat_asym) ** 2)
    sqnr_sym = compute_sqnr(data, r_hat_sym)
    sqnr_asym = compute_sqnr(data, r_hat_asym)

    print(f"\n{name}:")
    print(f"  Data range: [{data.min():.3f}, {data.max():.3f}]")
    print(f"  Symmetric:   s={s_sym:.6f}, z={z_sym}, MSE={mse_sym:.8f}, SQNR={sqnr_sym:.2f} dB")
    print(f"  Asymmetric:  s={s_asym:.6f}, z={z_asym}, MSE={mse_asym:.8f}, SQNR={sqnr_asym:.2f} dB")
    winner = "Asymmetric" if mse_asym < mse_sym else "Symmetric"
    print(f"  Winner: {winner} ({abs(sqnr_asym - sqnr_sym):.2f} dB advantage)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, data, title in [
    (axes[0], weights, 'Weights (Symmetric Data)'),
    (axes[1], activations, 'ReLU Activations (Asymmetric Data)'),
]:
    _, r_sym, _, _ = symmetric_quantize(data)
    _, r_asym, _, _ = asymmetric_quantize(data)

    err_sym = np.abs(data - r_sym)
    err_asym = np.abs(data - r_asym)

    ax.hist(err_sym, bins=50, alpha=0.6, color='#E74C3C', label=f'Symmetric (MSE={np.mean(err_sym**2):.6f})', edgecolor='black')
    ax.hist(err_asym, bins=50, alpha=0.6, color='#3498DB', label=f'Asymmetric (MSE={np.mean(err_asym**2):.6f})', edgecolor='black')
    ax.set_xlabel('Absolute Error', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-5'></a>
## Section 5: Per-Tensor vs Per-Channel Quantization

### Per-Tensor Quantization

A single $(s, z)$ pair is computed for the **entire tensor**. All elements share the same scale.

$$s = \frac{\max(r_{\text{all}}) - \min(r_{\text{all}})}{q_{\max} - q_{\min}}$$

### Per-Channel Quantization

Each output channel $c$ gets its own $(s_c, z_c)$. For a weight tensor $W \in \mathbb{R}^{C_{\text{out}} \times C_{\text{in}} \times k_H \times k_W}$:

$$s_c = \frac{\max(W_c) - \min(W_c)}{q_{\max} - q_{\min}}, \quad c = 0, 1, \ldots, C_{\text{out}}-1$$

This is critical because different channels can have very different value distributions:

```
 Per-tensor (one scale):                Per-channel (per-filter scale):

 Channel 0: [-0.1, 0.1] ─┐             Channel 0: [-0.1, 0.1] → s₀ = 0.0008
 Channel 1: [-0.3, 0.3]  ├─ s = 0.008  Channel 1: [-0.3, 0.3] → s₁ = 0.0024
 Channel 2: [-1.0, 1.0] ─┘             Channel 2: [-1.0, 1.0] → s₂ = 0.0078
                                             ▲
 Channel 0 uses only 25 of 256 bins!     Each channel uses full 256 bins
 → 10× worse resolution for ch 0        → optimal resolution per channel
```

In [ ]:
np.random.seed(42)
C_out, C_in, kH, kW = 32, 16, 3, 3

# Deliberately create channels with very different scales
W = np.random.randn(C_out, C_in, kH, kW).astype(np.float32)
channel_scales = np.logspace(-2, 0, C_out)  # scales from 0.01 to 1.0
W *= channel_scales.reshape(C_out, 1, 1, 1)

# Per-tensor quantization
s_tensor = (W.max() - W.min()) / 255
z_tensor = int(np.round(-W.min() / s_tensor))
W_q_tensor = np.clip(np.round(W / s_tensor) + z_tensor, 0, 255).astype(np.uint8)
W_deq_tensor = s_tensor * (W_q_tensor.astype(np.float32) - z_tensor)

# Per-channel quantization
W_deq_channel = np.zeros_like(W)
per_channel_scales = []
for c in range(C_out):
    w_c = W[c]
    s_c = (w_c.max() - w_c.min()) / 255
    if s_c < 1e-10:
        s_c = 1e-10
    z_c = int(np.round(-w_c.min() / s_c))
    q_c = np.clip(np.round(w_c / s_c) + z_c, 0, 255).astype(np.uint8)
    W_deq_channel[c] = s_c * (q_c.astype(np.float32) - z_c)
    per_channel_scales.append(s_c)

mse_tensor = np.mean((W - W_deq_tensor) ** 2)
mse_channel = np.mean((W - W_deq_channel) ** 2)

# Per-channel MSE per channel
mse_per_ch_tensor = [np.mean((W[c] - W_deq_tensor[c]) ** 2) for c in range(C_out)]
mse_per_ch_channel = [np.mean((W[c] - W_deq_channel[c]) ** 2) for c in range(C_out)]

print("Per-Tensor vs Per-Channel Quantization")
print("=" * 50)
print(f"Weight tensor shape: {W.shape}")
print(f"Channel value ranges: [{channel_scales.min():.4f}, {channel_scales.max():.4f}]")
print(f"\nPer-tensor MSE:  {mse_tensor:.8f}")
print(f"Per-channel MSE: {mse_channel:.8f}")
print(f"Improvement:     {mse_tensor / mse_channel:.1f}× lower MSE with per-channel")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

channels = list(range(C_out))
axes[0].bar(channels, mse_per_ch_tensor, alpha=0.7, label='Per-tensor', color='#E74C3C')
axes[0].bar(channels, mse_per_ch_channel, alpha=0.7, label='Per-channel', color='#2ECC71')
axes[0].set_xlabel('Output Channel', fontsize=11)
axes[0].set_ylabel('MSE (per channel)', fontsize=11)
axes[0].set_title('Quantization Error by Channel', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

axes[1].semilogy(channels, per_channel_scales, 'bo-', markersize=4)
axes[1].axhline(s_tensor, color='red', linestyle='--', linewidth=2, label=f'Per-tensor scale = {s_tensor:.5f}')
axes[1].set_xlabel('Output Channel', fontsize=11)
axes[1].set_ylabel('Scale (log)', fontsize=11)
axes[1].set_title('Scale Values: Per-Channel vs Per-Tensor', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: Static vs Dynamic Quantization

### Dynamic Quantization

Activation ranges are computed **at runtime** for each inference batch. Weights are pre-quantized offline.

$$s_{\text{act}} = \frac{\max(x_{\text{batch}}) - \min(x_{\text{batch}})}{q_{\max} - q_{\min}} \quad \text{(computed per forward pass)}$$

**Pros:** No calibration dataset needed, adapts to any input distribution.

**Cons:** Runtime overhead from min/max reduction over activations.

### Static Quantization

Activation ranges are determined **offline** using a calibration dataset. Ranges are baked into the model.

$$s_{\text{act}} = \frac{r_{\max}^{\text{(calibrated)}} - r_{\min}^{\text{(calibrated)}}}{q_{\max} - q_{\min}} \quad \text{(fixed after calibration)}$$

**Pros:** No runtime overhead, best latency.

**Cons:** Requires representative calibration data; accuracy degrades if deployment data differs.

### Comparison

| Property | Dynamic | Static |
|:---------|:--------|:-------|
| Activation ranges | Computed per-batch at runtime | Precomputed via calibration |
| Calibration needed | No | Yes |
| Runtime overhead | Higher (min/max reduction) | Lower (ranges baked in) |
| Weights quantized | Yes (offline) | Yes (offline) |
| Activations quantized | INT8 at runtime | INT8 at runtime |
| Best for | NLP / variable-length inputs | CNN / fixed-distribution inputs |

```
 ┌────────────────────────────────────────────────────────────────┐
 │            DYNAMIC QUANTIZATION                                │
 │                                                                │
 │  Weights: quantized offline (INT8)                             │
 │  Activations: quantized on-the-fly per batch                   │
 │                                                                │
 │  x_fp32 ──▶ [observe min/max] ──▶ [compute s,z] ──▶ q_int8    │
 │               ▲ runtime cost                                   │
 └────────────────────────────────────────────────────────────────┘

 ┌────────────────────────────────────────────────────────────────┐
 │            STATIC QUANTIZATION                                 │
 │                                                                │
 │  Weights: quantized offline (INT8)                             │
 │  Activations: ranges fixed from calibration                    │
 │                                                                │
 │  [Calibration]                                                 │
 │  calib_data ──▶ FP32 model ──▶ collect activation stats        │
 │                                  (min/max per tensor)          │
 │                                       │                        │
 │                                  [bake s, z into model]        │
 │                                       │                        │
 │  [Inference]                          ▼                        │
 │  x_fp32 ──▶ [use precomputed s,z] ──▶ q_int8                  │
 │               ▲ no runtime overhead                            │
 └────────────────────────────────────────────────────────────────┘
```

In [ ]:
np.random.seed(42)

# Simulate dynamic vs static quantization on a simple matmul
W = np.random.randn(128, 64).astype(np.float32) * 0.5

# Pre-quantize weights (shared by both approaches)
s_w, z_w = compute_scale_zp(float(W.min()), float(W.max()), -128, 127)
W_q = quantize(W, s_w, z_w, -128, 127)
W_deq = dequantize(W_q, s_w, z_w)

# Calibration data for static quantization
calib_batches = [np.random.randn(32, 128).astype(np.float32) * (0.5 + i * 0.1) for i in range(10)]
all_activations = np.concatenate([b @ W for b in calib_batches])
s_static, z_static = compute_scale_zp(float(all_activations.min()), float(all_activations.max()), -128, 127)

# Test on new data
test_batches = [np.random.randn(32, 128).astype(np.float32) * (0.5 + i * 0.1) for i in range(5)]

dynamic_errors = []
static_errors = []

for batch in test_batches:
    y_fp32 = batch @ W

    # Dynamic: compute activation scale on this batch
    y_approx = batch @ W_deq
    s_dyn, z_dyn = compute_scale_zp(float(y_approx.min()), float(y_approx.max()), -128, 127)
    y_q_dyn = quantize(y_approx, s_dyn, z_dyn, -128, 127)
    y_deq_dyn = dequantize(y_q_dyn, s_dyn, z_dyn)
    dynamic_errors.append(np.mean((y_fp32 - y_deq_dyn) ** 2))

    # Static: use pre-calibrated scale
    y_q_stat = quantize(y_approx, s_static, z_static, -128, 127)
    y_deq_stat = dequantize(y_q_stat, s_static, z_static)
    static_errors.append(np.mean((y_fp32 - y_deq_stat) ** 2))

print("Dynamic vs Static Quantization Simulation")
print("=" * 50)
print(f"{'Batch':>6} {'Dynamic MSE':>14} {'Static MSE':>14} {'Better':>10}")
print("-" * 50)
for i, (d, s_val) in enumerate(zip(dynamic_errors, static_errors)):
    better = 'Dynamic' if d < s_val else 'Static'
    print(f"{i:>6d} {d:>14.6f} {s_val:>14.6f} {better:>10}")
print(f"\nMean Dynamic MSE: {np.mean(dynamic_errors):.6f}")
print(f"Mean Static MSE:  {np.mean(static_errors):.6f}")

<a id='section-7'></a>
## Section 7: Post-Training Quantization (PTQ)

### Workflow

PTQ takes a fully trained FP32 model and quantizes it **without any retraining**:

```
 ┌──────────────────────────────────────────────────────────────────┐
 │                PTQ Workflow                                      │
 ├──────────────────────────────────────────────────────────────────┤
 │                                                                  │
 │  [1] Train model        FP32 model (full accuracy)               │
 │       │                                                          │
 │       ▼                                                          │
 │  [2] Calibrate          Run representative data through model    │
 │       │                 Collect activation statistics (optional)  │
 │       ▼                                                          │
 │  [3] Compute s, z       For each tensor (weights + activations)  │
 │       │                                                          │
 │  [4] Quantize           Replace FP32 tensors with INT8 + s, z    │
 │       │                                                          │
 │       ▼                                                          │
 │  [5] Validate           Compare accuracy vs FP32 baseline        │
 │                                                                  │
 └──────────────────────────────────────────────────────────────────┘
```

### Strengths
- No retraining needed — fast to apply
- Works well when weight distributions are well-behaved
- 4× model size reduction guaranteed

### Limitations
- Accuracy loss can be significant for sensitive models
- No way to compensate for quantization error during training
- Outlier weights can dominate the scale and waste dynamic range

In [ ]:
def build_mlp_onnx(input_dim=784, hidden_dim=256, output_dim=10):
    """Build a simple MLP as an ONNX model for quantization demos."""
    np.random.seed(42)
    initializers = []
    nodes = []

    W1 = numpy_helper.from_array(
        np.random.randn(input_dim, hidden_dim).astype(np.float32) * 0.01, 'W1')
    b1 = numpy_helper.from_array(
        np.zeros(hidden_dim, dtype=np.float32), 'b1')
    W2 = numpy_helper.from_array(
        np.random.randn(hidden_dim, hidden_dim).astype(np.float32) * 0.01, 'W2')
    b2 = numpy_helper.from_array(
        np.zeros(hidden_dim, dtype=np.float32), 'b2')
    W3 = numpy_helper.from_array(
        np.random.randn(hidden_dim, output_dim).astype(np.float32) * 0.01, 'W3')
    b3 = numpy_helper.from_array(
        np.zeros(output_dim, dtype=np.float32), 'b3')
    initializers.extend([W1, b1, W2, b2, W3, b3])

    nodes.append(helper.make_node('MatMul', ['X', 'W1'], ['h1_mm']))
    nodes.append(helper.make_node('Add', ['h1_mm', 'b1'], ['h1']))
    nodes.append(helper.make_node('Relu', ['h1'], ['a1']))
    nodes.append(helper.make_node('MatMul', ['a1', 'W2'], ['h2_mm']))
    nodes.append(helper.make_node('Add', ['h2_mm', 'b2'], ['h2']))
    nodes.append(helper.make_node('Relu', ['h2'], ['a2']))
    nodes.append(helper.make_node('MatMul', ['a2', 'W3'], ['h3_mm']))
    nodes.append(helper.make_node('Add', ['h3_mm', 'b3'], ['Y']))

    X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [None, input_dim])
    Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [None, output_dim])

    graph = helper.make_graph(nodes, 'mlp', [X_info], [Y_info], initializer=initializers)
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
    model.ir_version = 8
    check_model(model)
    return model


mlp_model = build_mlp_onnx()
fp32_path = '/tmp/mlp_fp32.onnx'
onnx.save(mlp_model, fp32_path)

print(f"MLP model built: {len(mlp_model.graph.node)} nodes")
print(f"FP32 model size: {os.path.getsize(fp32_path) / 1024:.1f} KB")
for n in mlp_model.graph.node:
    print(f"  {n.op_type}({', '.join(n.input)}) → {list(n.output)}")

<a id='section-8'></a>
## Section 8: Quantization-Aware Training (QAT) and the Straight-Through Estimator

### The Problem with Quantization Gradients

The quantization function $Q(r)$ involves rounding, which has **zero gradient almost everywhere**:

$$\frac{\partial Q}{\partial r} = \frac{\partial}{\partial r} \text{clamp}\left(\left\lfloor \frac{r}{s} \right\rceil + z\right) = 0 \quad \text{a.e.}$$

This makes standard backpropagation impossible through quantization nodes.

### The Straight-Through Estimator (STE)

Bengio et al. proposed the **Straight-Through Estimator**: pretend the quantization function is the identity during backpropagation:

$$\frac{\partial Q}{\partial r} \approx \begin{cases} 1 & \text{if } r \in [r_{\min}, r_{\max}] \\ 0 & \text{otherwise (clipped)} \end{cases}$$

### QAT with Fake-Quantization Nodes

During QAT, **fake-quant** nodes are inserted after weights and activations. In the forward pass, they simulate quantization error. In the backward pass, they use STE:

```
 ┌──────────────────────────────────────────────────────────────────┐
 │  FORWARD PASS (simulates quantization)                          │
 │                                                                  │
 │  W_fp32 ──▶ [FakeQuant] ──▶ W_fq ──▶ MatMul ──▶ output         │
 │              │                                                   │
 │              ├── quantize(W)  → q_int8                           │
 │              └── dequantize(q) → W_fq (FP32, but with           │
 │                                  quantization error baked in)    │
 ├──────────────────────────────────────────────────────────────────┤
 │  BACKWARD PASS (STE)                                            │
 │                                                                  │
 │  ∂L/∂W_fq ──▶ [STE: pass gradient through] ──▶ ∂L/∂W_fp32     │
 │                                                                  │
 │  The model learns to compensate for quantization error!          │
 └──────────────────────────────────────────────────────────────────┘
```

### PTQ vs QAT Comparison

```
 PTQ Pipeline:                          QAT Pipeline:
 ═══════════                            ═══════════

 ┌─────────────┐                        ┌─────────────────────────┐
 │ Train FP32  │                        │ Train FP32              │
 │ (standard)  │                        │ WITH fake-quant nodes   │
 └──────┬──────┘                        │ (STE for gradients)     │
        │                               └────────────┬────────────┘
        ▼                                            │
 ┌──────────────┐                                   ▼
 │ Calibrate    │  ← optional               ┌──────────────┐
 │ (static PTQ) │                           │ Remove fake-  │
 └──────┬───────┘                           │ quant nodes   │
        │                                    └──────┬───────┘
        ▼                                           │
 ┌──────────────┐                                   ▼
 │ Quantize     │                           ┌──────────────┐
 │ weights/acts │                           │ Export INT8   │
 └──────┬───────┘                           │ model         │
        │                                    └──────┬───────┘
        ▼                                           │
 ┌──────────────┐                                   ▼
 │ Validate     │                           ┌──────────────┐
 └──────────────┘                           │ Validate     │
                                             └──────────────┘
  ✓ Fast (no retrain)                       ✓ Best INT8 accuracy
  ✗ May lose accuracy                       ✗ Requires training
```

In [ ]:
def fake_quantize(x, bits=8):
    """Simulate quantization in the forward pass (fake-quant node)."""
    q_max = (2 ** (bits - 1)) - 1
    q_min = -(2 ** (bits - 1))
    x_max = np.max(np.abs(x))
    if x_max < 1e-10:
        return x.copy()
    s = x_max / q_max
    x_q = np.clip(np.round(x / s), q_min, q_max)
    return (x_q * s).astype(np.float32)  # dequantize back to FP32


def ste_gradient(grad_output, x, bits=8):
    """STE: pass gradient through if x is within quantization range."""
    q_max = (2 ** (bits - 1)) - 1
    x_max = np.max(np.abs(x))
    mask = (np.abs(x) <= x_max).astype(np.float32)
    return grad_output * mask


# Demonstrate fake quantization effect
np.random.seed(42)
W_original = np.random.randn(64, 32).astype(np.float32) * 0.5
W_fakequant = fake_quantize(W_original, bits=8)

print("Fake Quantization (simulating QAT forward pass)")
print("=" * 50)
print(f"Original weights range: [{W_original.min():.4f}, {W_original.max():.4f}]")
print(f"Fake-quantized range:   [{W_fakequant.min():.4f}, {W_fakequant.max():.4f}]")
print(f"MSE(W, FakeQ(W)):       {np.mean((W_original - W_fakequant)**2):.8f}")
print(f"Unique FP32 values:     {len(np.unique(W_original))} (all unique)")
print(f"Unique FakeQ values:    {len(np.unique(W_fakequant))} (quantization grid)")

# STE gradient demo
grad_out = np.ones_like(W_original)
grad_ste = ste_gradient(grad_out, W_original, bits=8)
print(f"\nSTE gradient pass-through rate: {np.mean(grad_ste):.4f} (should be ~1.0 for in-range values)")
print(f"STE preserves gradient shape: {grad_ste.shape == grad_out.shape}")

In [ ]:
# Visualize fake quantization effect at different bit widths
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

test_weights = np.random.randn(10000).astype(np.float32) * 0.5

for idx, bits in enumerate([2, 3, 4, 6, 8, 16]):
    ax = axes[idx // 3, idx % 3]
    fq = fake_quantize(test_weights, bits=bits)
    mse = np.mean((test_weights - fq) ** 2)
    sqnr = compute_sqnr(test_weights, fq)

    ax.hist(test_weights, bins=80, alpha=0.5, color='blue', label='Original', density=True)
    ax.hist(fq, bins=80, alpha=0.5, color='red', label='Fake-quantized', density=True)
    ax.set_title(f'{bits}-bit: MSE={mse:.6f}, SQNR={sqnr:.1f}dB', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xlabel('Value')

plt.suptitle('Effect of Fake Quantization at Different Bit Widths', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Calibration Methods

Static quantization requires choosing $r_{\min}$ and $r_{\max}$ for each activation tensor. The calibration method determines how these bounds are selected.

### Method 1: MinMax

Use the observed minimum and maximum across calibration data:

$$r_{\min} = \min_{i} x_i^{(\text{calib})}, \quad r_{\max} = \max_{i} x_i^{(\text{calib})}$$

**Pros:** Simple, captures full range. **Cons:** Sensitive to outliers.

### Method 2: Percentile

Clip at the $p$-th and $(100-p)$-th percentiles (e.g., $p=0.1$):

$$r_{\min} = P_{0.1\%}(x), \quad r_{\max} = P_{99.9\%}(x)$$

**Pros:** Robust to outliers. **Cons:** Clips extreme values.

### Method 3: Entropy (KL Divergence)

Find the clipping threshold $t$ that minimizes the KL divergence between the original and quantized distributions:

$$t^* = \arg\min_t D_{\text{KL}}\left(P_{\text{original}} \| Q_{\text{quantized}}(t)\right)$$

$$D_{\text{KL}}(P \| Q) = \sum_i P(i) \log \frac{P(i)}{Q(i)}$$

This is the method used by TensorRT and is available in ONNX Runtime.

### Method 4: MSE Minimization

Find the clipping range that minimizes mean squared error:

$$t^* = \arg\min_t \mathbb{E}\left[(x - D(Q(x; t)))^2\right]$$

### Calibration Pipeline

```
 ┌──────────────────────────────────────────────────────────────────────┐
 │                    CALIBRATION PIPELINE                              │
 ├──────────────────────────────────────────────────────────────────────┤
 │                                                                      │
 │  Representative Data                                                 │
 │  (100-1000 samples)                                                  │
 │       │                                                              │
 │       ▼                                                              │
 │  ┌───────────────────────────────────────┐                           │
 │  │ Instrumented FP32 Model               │                           │
 │  │ (hooks collect activation statistics)  │                           │
 │  └────────────────┬──────────────────────┘                           │
 │                   │                                                  │
 │       ┌───────────┼───────────┬────────────────┐                     │
 │       ▼           ▼           ▼                ▼                     │
 │   ┌────────┐  ┌────────┐  ┌──────────┐  ┌──────────┐               │
 │   │ MinMax │  │Percentl│  │ Entropy  │  │ MSE Min  │               │
 │   │        │  │  99.9% │  │ KL-div   │  │          │               │
 │   └───┬────┘  └───┬────┘  └────┬─────┘  └────┬─────┘               │
 │       │           │            │              │                      │
 │       └───────────┴────────────┴──────────────┘                      │
 │                         │                                            │
 │                    (s, z) per tensor                                  │
 │                         │                                            │
 │                         ▼                                            │
 │              Quantized ONNX Model                                    │
 │              (QDQ nodes inserted)                                    │
 │                                                                      │
 └──────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def calibrate_minmax(data):
    return float(data.min()), float(data.max())


def calibrate_percentile(data, percentile=99.9):
    low = np.percentile(data, 100 - percentile)
    high = np.percentile(data, percentile)
    return float(low), float(high)


def calibrate_mse(data, n_steps=200, bits=8):
    """Find clipping threshold that minimizes quantization MSE."""
    best_mse = float('inf')
    best_t = float(np.max(np.abs(data)))
    q_max = (2 ** (bits - 1)) - 1
    q_min = -(2 ** (bits - 1))

    for i in range(1, n_steps + 1):
        t = float(np.max(np.abs(data))) * i / n_steps
        clipped = np.clip(data, -t, t)
        s = 2 * t / (q_max - q_min)
        if s < 1e-12:
            continue
        q = np.clip(np.round(clipped / s), q_min, q_max)
        deq = q * s
        mse = np.mean((data - deq) ** 2)
        if mse < best_mse:
            best_mse = mse
            best_t = t

    return float(-best_t), float(best_t)


def calibrate_entropy(data, n_bins=2048, n_steps=200, bits=8):
    """Find clipping threshold that minimizes KL divergence."""
    hist, bin_edges = np.histogram(data, bins=n_bins)
    hist = hist.astype(np.float64)
    hist += 1e-10

    n_quant_bins = 2 ** bits
    best_kl = float('inf')
    best_idx = n_bins

    for i in range(n_quant_bins, n_bins + 1):
        p = hist[:i].copy()
        p /= p.sum()

        # Create quantized distribution
        stride = i / n_quant_bins
        q = np.zeros(i)
        for j in range(n_quant_bins):
            start = int(round(j * stride))
            end = int(round((j + 1) * stride))
            end = min(end, i)
            if start >= end:
                continue
            total = p[start:end].sum()
            if total > 0:
                q[start:end] = total / (end - start)

        q += 1e-10
        q /= q.sum()

        kl = np.sum(p * np.log(p / q))
        if kl < best_kl:
            best_kl = kl
            best_idx = i

    threshold = bin_edges[best_idx]
    return float(-threshold), float(threshold)


# Generate realistic activation data with outliers
np.random.seed(42)
normal_data = np.random.randn(50000).astype(np.float32)
outliers = np.array([10.0, -12.0, 8.5, -9.0, 11.0], dtype=np.float32)
activation_data = np.concatenate([normal_data, outliers])

methods = {
    'MinMax': calibrate_minmax(activation_data),
    'Percentile (99.9%)': calibrate_percentile(activation_data, 99.9),
    'MSE Minimization': calibrate_mse(activation_data),
    'Entropy (KL-div)': calibrate_entropy(activation_data),
}

print("Calibration Methods Comparison")
print("=" * 70)
print(f"Data: {len(activation_data)} samples, range [{activation_data.min():.2f}, {activation_data.max():.2f}]")
print(f"Outliers injected: {outliers}")
print(f"")
print(f"{'Method':<22} {'r_min':>8} {'r_max':>8} {'Scale':>10} {'Quant MSE':>12} {'SQNR (dB)':>10}")
print("-" * 70)

for name, (r_lo, r_hi) in methods.items():
    s_cal, z_cal = compute_scale_zp(r_lo, r_hi, -128, 127)
    clipped = np.clip(activation_data, r_lo, r_hi)
    q_cal = quantize(clipped, s_cal, z_cal, -128, 127)
    r_hat_cal = dequantize(q_cal, s_cal, z_cal)
    mse_cal = np.mean((activation_data - r_hat_cal) ** 2)
    sqnr_cal = compute_sqnr(activation_data, r_hat_cal)
    print(f"{name:<22} {r_lo:>8.3f} {r_hi:>8.3f} {s_cal:>10.6f} {mse_cal:>12.8f} {sqnr_cal:>10.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Activation distribution with calibration thresholds
axes[0].hist(activation_data, bins=200, color='#3498DB', alpha=0.7, edgecolor='black', linewidth=0.3, density=True)
colors = ['red', 'green', 'orange', 'purple']
for (name, (r_lo, r_hi)), color in zip(methods.items(), colors):
    axes[0].axvline(r_lo, color=color, linestyle='--', linewidth=2, label=f'{name}: [{r_lo:.2f}, {r_hi:.2f}]')
    axes[0].axvline(r_hi, color=color, linestyle='--', linewidth=2)
axes[0].set_xlabel('Activation Value', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Calibration Thresholds on Activation Distribution', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='upper left')
axes[0].grid(alpha=0.3)

# Plot 2: SQNR comparison
method_names = list(methods.keys())
sqnr_list = []
for name, (r_lo, r_hi) in methods.items():
    s_cal, z_cal = compute_scale_zp(r_lo, r_hi, -128, 127)
    clipped = np.clip(activation_data, r_lo, r_hi)
    q_cal = quantize(clipped, s_cal, z_cal, -128, 127)
    r_hat_cal = dequantize(q_cal, s_cal, z_cal)
    sqnr_list.append(compute_sqnr(activation_data, r_hat_cal))

bars = axes[1].barh(method_names, sqnr_list, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_xlabel('SQNR (dB) — higher is better', fontsize=11)
axes[1].set_title('Calibration Method Quality (SQNR)', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, sqnr_list):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f} dB', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

<a id='section-10'></a>
## Section 10: ONNX Quantization Toolchain and QDQ Format

### The QDQ Representation

ONNX uses **QuantizeLinear** (Q) and **DequantizeLinear** (DQ) operators to represent quantized models in a framework-agnostic way. This is called the **QDQ format**.

```
 ┌──────────────────────────────────────────────────────────────────────┐
 │                  QDQ Node Insertion Pattern                          │
 ├──────────────────────────────────────────────────────────────────────┤
 │                                                                      │
 │  BEFORE (FP32):                                                      │
 │                                                                      │
 │    x_fp32 ──▶ Conv(W_fp32) ──▶ y_fp32                               │
 │                                                                      │
 │  AFTER (QDQ):                                                        │
 │                                                                      │
 │    x_fp32 ──▶ Q ──▶ DQ ──▶ ┐                                        │
 │                            ├──▶ Conv ──▶ Q ──▶ DQ ──▶ y_fp32        │
 │    W_fp32 ──▶ Q ──▶ DQ ──▶ ┘                                        │
 │                                                                      │
 │  Where:                                                              │
 │    Q  = QuantizeLinear(x, scale, zero_point) → uint8/int8            │
 │    DQ = DequantizeLinear(q, scale, zero_point) → float32             │
 │                                                                      │
 │  At runtime, the execution provider (EP) can:                        │
 │    1. Recognize Q-Conv-DQ pattern                                    │
 │    2. Fuse into a single INT8 Conv kernel                            │
 │    3. Execute entirely in INT8 (except final DQ)                     │
 │                                                                      │
 │  ┌─────────┐   ┌─────────┐   ┌─────────┐   ┌─────────┐             │
 │  │ Q(input)│──▶│DQ(input)│──▶│  Conv   │──▶│Q(output)│──▶ ...      │
 │  │ s₁, z₁  │   │ s₁, z₁  │   │(INT8 W) │   │ s₂, z₂  │             │
 │  └─────────┘   └─────────┘   └─────────┘   └─────────┘             │
 │       ▲              ▲              ▲              ▲                 │
 │    FP32→INT8     INT8→FP32     Fused by EP     FP32→INT8            │
 │                                                                      │
 └──────────────────────────────────────────────────────────────────────┘
```

### ONNX Runtime Quantization API

| Function | Description |
|:---------|:------------|
| `quantize_dynamic(model_in, model_out, weight_type)` | Dynamic PTQ — weights quantized offline, activations at runtime |
| `quantize_static(model_in, model_out, calibration_reader, ...)` | Static PTQ — requires calibration data reader |
| `QuantType.QInt8` / `QuantType.QUInt8` | Target integer type |
| `CalibrationMethod.MinMax` / `.Entropy` / `.Percentile` | Calibration strategy for static quantization |

In [ ]:
# Build a QDQ representation manually to understand the format
np.random.seed(42)

# Original Conv weights
W_data = np.random.randn(16, 3, 3, 3).astype(np.float32) * 0.1

# Compute quantization parameters for weights
w_scale = (W_data.max() - W_data.min()) / 255.0
w_zp = np.clip(np.round(-W_data.min() / w_scale), 0, 255).astype(np.uint8)

# Compute quantization parameters for input (assume calibrated range)
in_scale = np.float32(0.05)
in_zp = np.uint8(128)

# Compute quantization parameters for output
out_scale = np.float32(0.1)
out_zp = np.uint8(128)

# Build QDQ model
W_init = numpy_helper.from_array(W_data, 'W')
w_scale_init = numpy_helper.from_array(np.float32(w_scale), 'w_scale')
w_zp_init = numpy_helper.from_array(w_zp, 'w_zp')
in_scale_init = numpy_helper.from_array(in_scale, 'in_scale')
in_zp_init = numpy_helper.from_array(in_zp, 'in_zp')
out_scale_init = numpy_helper.from_array(out_scale, 'out_scale')
out_zp_init = numpy_helper.from_array(out_zp, 'out_zp')

nodes = [
    helper.make_node('QuantizeLinear', ['X', 'in_scale', 'in_zp'], ['X_q'], name='Q_input'),
    helper.make_node('DequantizeLinear', ['X_q', 'in_scale', 'in_zp'], ['X_dq'], name='DQ_input'),
    helper.make_node('QuantizeLinear', ['W', 'w_scale', 'w_zp'], ['W_q'], name='Q_weight'),
    helper.make_node('DequantizeLinear', ['W_q', 'w_scale', 'w_zp'], ['W_dq'], name='DQ_weight'),
    helper.make_node('Conv', ['X_dq', 'W_dq'], ['Y_fp'], name='Conv', kernel_shape=[3, 3], pads=[1,1,1,1]),
    helper.make_node('QuantizeLinear', ['Y_fp', 'out_scale', 'out_zp'], ['Y_q'], name='Q_output'),
    helper.make_node('DequantizeLinear', ['Y_q', 'out_scale', 'out_zp'], ['Y'], name='DQ_output'),
]

X_info = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 32, 32])
Y_info = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 16, 32, 32])

graph = helper.make_graph(
    nodes, 'qdq_conv',
    inputs=[X_info], outputs=[Y_info],
    initializer=[W_init, w_scale_init, w_zp_init, in_scale_init, in_zp_init, out_scale_init, out_zp_init]
)
qdq_model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
qdq_model.ir_version = 8

print("QDQ Model Structure")
print("=" * 60)
for n in qdq_model.graph.node:
    print(f"  [{n.name:>12}] {n.op_type:>18}({', '.join(n.input)}) → {list(n.output)}")
print(f"\nTotal nodes: {len(qdq_model.graph.node)}")
print(f"Pattern: Q(input) → DQ → Conv(DQ(Q(W))) → Q(output) → DQ")
print(f"\nThe EP can fuse Q-DQ-Conv-Q-DQ into a single INT8 Conv kernel.")

<a id='section-11'></a>
## Section 11: Dynamic Quantization with ONNX Runtime

Dynamic quantization is the simplest quantization method to apply: it quantizes model weights offline and computes activation ranges at runtime. No calibration data is required.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

fp32_path = '/tmp/mlp_fp32.onnx'
dyn_int8_path = '/tmp/mlp_dynamic_int8.onnx'

# Apply dynamic quantization
quantize_dynamic(
    model_input=fp32_path,
    model_output=dyn_int8_path,
    weight_type=QuantType.QInt8,
)

# Compare
fp32_size = os.path.getsize(fp32_path)
int8_size = os.path.getsize(dyn_int8_path)

print("Dynamic Quantization Results")
print("=" * 50)
print(f"FP32 model size:  {fp32_size / 1024:.1f} KB")
print(f"INT8 model size:  {int8_size / 1024:.1f} KB")
print(f"Compression:      {fp32_size / int8_size:.2f}×")

# Load and inspect quantized model
dyn_model = onnx.load(dyn_int8_path)
print(f"\nQuantized model nodes: {len(dyn_model.graph.node)}")
op_counts = {}
for n in dyn_model.graph.node:
    op_counts[n.op_type] = op_counts.get(n.op_type, 0) + 1
for op, count in sorted(op_counts.items()):
    print(f"  {op}: {count}")

# Verify numerical similarity
sess_fp32 = ort.InferenceSession(fp32_path)
sess_int8 = ort.InferenceSession(dyn_int8_path)

np.random.seed(0)
max_diffs = []
for _ in range(50):
    x_test = np.random.randn(4, 784).astype(np.float32)
    y_fp32 = sess_fp32.run(None, {'X': x_test})[0]
    y_int8 = sess_int8.run(None, {'X': x_test})[0]
    max_diffs.append(np.max(np.abs(y_fp32 - y_int8)))

print(f"\nNumerical Comparison (50 random inputs):")
print(f"  Max |FP32 - INT8|: {np.max(max_diffs):.6f}")
print(f"  Mean |FP32 - INT8|: {np.mean(max_diffs):.6f}")

<a id='section-12'></a>
## Section 12: Static Quantization with ONNX Runtime

Static quantization requires a **calibration data reader** that provides representative input tensors. The quantizer runs the model on this data to collect activation statistics and compute optimal scale/zero-point values.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationMethod, QuantFormat
from onnxruntime.quantization import CalibrationDataReader


class NumpyCalibrationReader(CalibrationDataReader):
    """Feeds numpy arrays as calibration data."""
    def __init__(self, data_list, input_name='X'):
        self.data_list = data_list
        self.input_name = input_name
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.data_list):
            return None
        batch = self.data_list[self.idx]
        self.idx += 1
        return {self.input_name: batch}


# Prepare model (ORT static quant needs pre-processed model)
from onnxruntime.quantization import shape_inference

fp32_path = '/tmp/mlp_fp32.onnx'
preprocessed_path = '/tmp/mlp_preprocessed.onnx'

shape_inference.quant_pre_process(fp32_path, preprocessed_path)

# Generate calibration data
np.random.seed(42)
calib_data = [np.random.randn(8, 784).astype(np.float32) for _ in range(20)]

# Quantize with different calibration methods
results = {}
for method_name, method_enum in [
    ('MinMax', CalibrationMethod.MinMax),
    ('Entropy', CalibrationMethod.Entropy),
    ('Percentile', CalibrationMethod.Percentile),
]:
    reader = NumpyCalibrationReader(calib_data)
    output_path = f'/tmp/mlp_static_{method_name.lower()}.onnx'

    try:
        quantize_static(
            model_input=preprocessed_path,
            model_output=output_path,
            calibration_data_reader=reader,
            calibrate_method=method_enum,
            quant_format=QuantFormat.QDQ,
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QUInt8,
        )
        size = os.path.getsize(output_path)

        # Test accuracy
        sess = ort.InferenceSession(output_path)
        diffs = []
        for _ in range(50):
            x = np.random.randn(4, 784).astype(np.float32)
            y_ref = sess_fp32.run(None, {'X': x})[0]
            y_q = sess.run(None, {'X': x})[0]
            diffs.append(np.max(np.abs(y_ref - y_q)))

        results[method_name] = {
            'size_kb': size / 1024,
            'max_diff': np.max(diffs),
            'mean_diff': np.mean(diffs),
        }
    except Exception as e:
        results[method_name] = {'error': str(e)}

print("Static Quantization Results (by Calibration Method)")
print("=" * 65)
print(f"FP32 baseline: {os.path.getsize(fp32_path) / 1024:.1f} KB")
print(f"")
print(f"{'Method':<15} {'Size (KB)':>10} {'Max Diff':>12} {'Mean Diff':>12}")
print("-" * 65)
for name, res in results.items():
    if 'error' in res:
        print(f"{name:<15} Error: {res['error'][:40]}")
    else:
        print(f"{name:<15} {res['size_kb']:>10.1f} {res['max_diff']:>12.6f} {res['mean_diff']:>12.6f}")

<a id='section-13'></a>
## Section 13: Model Size and Performance Comparison

Let's do a comprehensive comparison of the original FP32 model versus all quantized variants.

In [ ]:
import time

def benchmark_model(model_path, input_shape=(4, 784), n_warmup=10, n_runs=100):
    """Measure inference latency."""
    sess = ort.InferenceSession(model_path)
    input_name = sess.get_inputs()[0].name
    x = np.random.randn(*input_shape).astype(np.float32)

    for _ in range(n_warmup):
        sess.run(None, {input_name: x})

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {input_name: x})
        times.append((time.perf_counter() - t0) * 1000)

    return {
        'median_ms': np.median(times),
        'p95_ms': np.percentile(times, 95),
        'mean_ms': np.mean(times),
    }


models = {
    'FP32': fp32_path,
    'Dynamic INT8': dyn_int8_path,
}

# Add static models if they exist
for method in ['minmax', 'entropy', 'percentile']:
    path = f'/tmp/mlp_static_{method}.onnx'
    if os.path.exists(path):
        models[f'Static INT8 ({method})'] = path

print("Model Size and Latency Comparison")
print("=" * 80)
print(f"{'Model':<25} {'Size (KB)':>10} {'Compress':>10} {'Median (ms)':>12} {'P95 (ms)':>10}")
print("-" * 80)

fp32_size = os.path.getsize(fp32_path) / 1024
comparison_data = []

for name, path in models.items():
    size_kb = os.path.getsize(path) / 1024
    compress = fp32_size / size_kb
    perf = benchmark_model(path)
    comparison_data.append((name, size_kb, compress, perf['median_ms'], perf['p95_ms']))
    print(f"{name:<25} {size_kb:>10.1f} {compress:>9.2f}× {perf['median_ms']:>12.3f} {perf['p95_ms']:>10.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

names = [d[0] for d in comparison_data]
sizes = [d[1] for d in comparison_data]
compressions = [d[2] for d in comparison_data]
latencies = [d[3] for d in comparison_data]

colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

# Plot 1: Model sizes
bars = axes[0].bar(range(len(names)), sizes, color=colors, edgecolor='black', linewidth=1.2)
axes[0].set_xticks(range(len(names)))
axes[0].set_xticklabels(names, rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Size (KB)', fontsize=11)
axes[0].set_title('Model Size Comparison', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for bar, sz in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{sz:.0f}', ha='center', fontsize=9, fontweight='bold')

# Plot 2: Compression ratio
bars2 = axes[1].bar(range(len(names)), compressions, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_xticks(range(len(names)))
axes[1].set_xticklabels(names, rotation=25, ha='right', fontsize=9)
axes[1].set_ylabel('Compression Ratio', fontsize=11)
axes[1].set_title('Compression vs FP32', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for bar, c in zip(bars2, compressions):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{c:.2f}×', ha='center', fontsize=9, fontweight='bold')

# Plot 3: Latency
bars3 = axes[2].bar(range(len(names)), latencies, color=colors, edgecolor='black', linewidth=1.2)
axes[2].set_xticks(range(len(names)))
axes[2].set_xticklabels(names, rotation=25, ha='right', fontsize=9)
axes[2].set_ylabel('Latency (ms)', fontsize=11)
axes[2].set_title('Inference Latency (median)', fontsize=12, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
for bar, lat in zip(bars3, latencies):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{lat:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

<a id='section-14'></a>
## Section 14: FP32 vs FP16 vs INT8 — Comprehensive Comparison

### Data Type Properties

| Property | FP32 | FP16 | BF16 | INT8 |
|:---------|:----:|:----:|:----:|:----:|
| Bits per element | 32 | 16 | 16 | 8 |
| Exponent bits | 8 | 5 | 8 | N/A |
| Mantissa bits | 23 | 10 | 7 | N/A |
| Dynamic range | $\pm 3.4 \times 10^{38}$ | $\pm 6.5 \times 10^{4}$ | $\pm 3.4 \times 10^{38}$ | $[-128, 127]$ |
| Precision | ~7 decimal digits | ~3 decimal digits | ~2 decimal digits | 256 levels |
| Model size (rel.) | 1× | 0.5× | 0.5× | 0.25× |
| Memory bandwidth | 1× | 0.5× | 0.5× | 0.25× |
| Compute throughput | 1× | 2× (GPU) | 2× (GPU) | 2-4× (with INT8 kernels) |
| Training support | Full | Mixed precision | Mixed precision | QAT only |
| Accuracy impact | Baseline | Minimal | Small | Varies (needs calibration) |

### Bit Layout

```
 FP32 (32 bits):
 ┌─┬──────────┬───────────────────────────────────────┐
 │S│ Exponent │           Mantissa                     │
 │1│  8 bits  │          23 bits                       │
 └─┴──────────┴───────────────────────────────────────┘

 FP16 (16 bits):
 ┌─┬───────┬──────────────────┐
 │S│ Exp   │    Mantissa      │
 │1│5 bits │   10 bits        │
 └─┴───────┴──────────────────┘

 BF16 (16 bits):
 ┌─┬──────────┬─────────┐
 │S│ Exponent │ Mantissa│
 │1│  8 bits  │ 7 bits  │
 └─┴──────────┴─────────┘

 INT8 (8 bits):
 ┌─┬─────────────────┐
 │S│  Magnitude      │  (two's complement: -128 to 127)
 │1│   7 bits        │
 └─┴─────────────────┘
```

### Decision Framework

```
                   Need quantized model?
                          │
             ┌────────────┼────────────┐
             ▼            │            ▼
            No            │         Yes
             │            │            │
        Use FP32          │     ┌──────┴──────┐
        or FP16           │     │             │
                          │  Can retrain?  No retrain
                          │     │             │
                          │  Use QAT       ┌──┴──┐
                          │  (best acc.)   │     │
                          │             Have    No calib
                          │            calib    data
                          │            data?      │
                          │              │     Use Dynamic
                          │           Use        PTQ
                          │          Static
                          │           PTQ
```

In [ ]:
# Demonstrate FP32 vs FP16 conversion and size comparison
from onnx import version_converter

fp32_model = onnx.load(fp32_path)

# Convert to FP16
try:
    from onnxruntime.transformers import float16
    fp16_model = float16.convert_float_to_float16(onnx.load(fp32_path))
    fp16_path = '/tmp/mlp_fp16.onnx'
    onnx.save(fp16_model, fp16_path)
    fp16_available = True
except ImportError:
    fp16_available = False
    # Manual FP16 conversion for initializers
    fp16_model = onnx.load(fp32_path)
    for init in fp16_model.graph.initializer:
        if init.data_type == TensorProto.FLOAT:
            arr = numpy_helper.to_array(init).astype(np.float16)
            new_init = numpy_helper.from_array(arr, init.name)
            init.CopyFrom(new_init)
    fp16_path = '/tmp/mlp_fp16.onnx'
    onnx.save(fp16_model, fp16_path)
    fp16_available = True

print("FP32 vs FP16 vs INT8 Size Comparison")
print("=" * 50)

all_models = {'FP32': fp32_path, 'Dynamic INT8': dyn_int8_path}
if fp16_available:
    all_models['FP16'] = fp16_path
for method in ['minmax', 'entropy', 'percentile']:
    path = f'/tmp/mlp_static_{method}.onnx'
    if os.path.exists(path):
        all_models[f'Static INT8 ({method})'] = path

base_size = os.path.getsize(fp32_path)
for name, path in all_models.items():
    sz = os.path.getsize(path)
    ratio = base_size / sz
    print(f"  {name:<25} {sz/1024:>8.1f} KB  ({ratio:.2f}× compression)")

# Weight distribution comparison
def get_weight_stats(model_path):
    model = onnx.load(model_path)
    all_weights = []
    for init in model.graph.initializer:
        try:
            arr = numpy_helper.to_array(init).flatten().astype(np.float32)
            all_weights.extend(arr.tolist())
        except:
            pass
    return np.array(all_weights)

w_fp32 = get_weight_stats(fp32_path)
w_int8 = get_weight_stats(dyn_int8_path)

print(f"\nWeight Statistics:")
print(f"  FP32: {len(w_fp32)} params, unique={len(np.unique(w_fp32))}, range=[{w_fp32.min():.4f}, {w_fp32.max():.4f}]")
print(f"  INT8: {len(w_int8)} values, unique={len(np.unique(w_int8))}, range=[{w_int8.min():.4f}, {w_int8.max():.4f}]")

In [ ]:
# Final comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Weight distributions
axes[0, 0].hist(w_fp32, bins=100, alpha=0.6, color='#2196F3', label='FP32 weights', density=True, edgecolor='black', linewidth=0.3)
axes[0, 0].hist(w_int8[w_int8 != 0], bins=100, alpha=0.6, color='#FF5722', label='INT8 weights (dequantized)', density=True, edgecolor='black', linewidth=0.3)
axes[0, 0].set_xlabel('Weight Value', fontsize=11)
axes[0, 0].set_ylabel('Density', fontsize=11)
axes[0, 0].set_title('Weight Distribution: FP32 vs INT8', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Plot 2: Quantization error across bit widths (bar chart)
bit_range = [2, 4, 8, 16, 32]
size_bytes = [b / 8 for b in bit_range]
sqnr_approx = [6.02 * b + 1.76 for b in bit_range]
bar_colors = ['#E74C3C', '#F39C12', '#2ECC71', '#3498DB', '#9B59B6']

ax2 = axes[0, 1]
bars = ax2.bar(range(len(bit_range)), sqnr_approx, color=bar_colors, edgecolor='black', linewidth=1.2)
ax2.set_xticks(range(len(bit_range)))
ax2.set_xticklabels([f'{b}-bit' for b in bit_range], fontsize=10)
ax2.set_ylabel('SQNR (dB)', fontsize=11)
ax2.set_title('Theoretical SQNR by Precision', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, sqnr_approx):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

# Plot 3: Size vs accuracy tradeoff
precisions = ['FP32\n(baseline)', 'FP16', 'INT8\n(dynamic)', 'INT8\n(static)']
relative_sizes = [1.0, 0.5, 0.25, 0.25]
relative_accuracy = [1.0, 0.999, 0.995, 0.997]

scatter_colors = ['#3498DB', '#2ECC71', '#E74C3C', '#F39C12']
for i, (sz, acc, label) in enumerate(zip(relative_sizes, relative_accuracy, precisions)):
    axes[1, 0].scatter(sz, acc, s=200, c=scatter_colors[i], edgecolors='black', linewidth=1.5, zorder=5)
    axes[1, 0].annotate(label, (sz, acc), textcoords='offset points', xytext=(10, -5), fontsize=9)

axes[1, 0].set_xlabel('Relative Model Size', fontsize=11)
axes[1, 0].set_ylabel('Relative Accuracy', fontsize=11)
axes[1, 0].set_title('Size vs Accuracy Tradeoff (Typical)', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].set_xlim(-0.05, 1.15)
axes[1, 0].set_ylim(0.99, 1.005)

# Plot 4: Memory bandwidth savings
dtypes = ['FP32', 'FP16/BF16', 'INT8']
bytes_per_elem = [4, 2, 1]
throughput_mult = [1, 2, 4]

x_pos = np.arange(len(dtypes))
width = 0.35
bars1 = axes[1, 1].bar(x_pos - width/2, bytes_per_elem, width, label='Bytes/element', color='#3498DB', edgecolor='black')
bars2 = axes[1, 1].bar(x_pos + width/2, throughput_mult, width, label='Throughput multiplier', color='#E74C3C', edgecolor='black')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(dtypes, fontsize=11)
axes[1, 1].set_ylabel('Value', fontsize=11)
axes[1, 1].set_title('Memory Footprint vs Throughput', fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

<a id='section-15'></a>
## Section 15: Summary

### Key Formulas

| Formula | Expression |
|:--------|:-----------|
| Quantize | $Q(r) = \text{clamp}(\lfloor r/s \rceil + z, q_{\min}, q_{\max})$ |
| Dequantize | $D(q) = s \cdot (q - z)$ |
| Scale | $s = \frac{r_{\max} - r_{\min}}{q_{\max} - q_{\min}}$ |
| Zero-point | $z = q_{\min} - \lfloor r_{\min}/s \rceil$ |
| Error bound | $|r - D(Q(r))| \leq s/2$ |
| MSE (uniform) | $\text{MSE} = s^2/12$ |
| SQNR | $\text{SQNR} = 10\log_{10}(\sigma_r^2 / (s^2/12)) \approx 6.02n + 1.76$ dB |
| STE gradient | $\partial Q / \partial r \approx 1$ (in range) |

### Method Selection Guide

| Scenario | Recommended Method |
|:---------|:-------------------|
| Quick deployment, no calibration data | Dynamic PTQ |
| Have calibration data, need best latency | Static PTQ |
| PTQ accuracy unacceptable | QAT |
| GPU inference, moderate compression | FP16 |
| Edge/mobile deployment | INT8 (static PTQ or QAT) |
| Research / extreme compression | INT4 / INT2 |

### ONNX Quantization Toolchain

| API | Use Case |
|:----|:---------|
| `quantize_dynamic()` | Dynamic PTQ — no calibration |
| `quantize_static()` | Static PTQ — needs calibration reader |
| `CalibrationMethod.MinMax` | Simple, outlier-sensitive |
| `CalibrationMethod.Entropy` | KL-divergence, best for distributions with tails |
| `CalibrationMethod.Percentile` | Robust to outliers |
| `QuantFormat.QDQ` | QDQ node format (recommended) |

### Key Takeaways

1. **Quantization reduces model size 2-4× and improves throughput** with minimal accuracy loss when done correctly
2. **Per-channel quantization** is essential for models with heterogeneous weight distributions
3. **Calibration data quality** is the single most important factor for static PTQ accuracy
4. **The STE** enables QAT by providing a gradient approximation through the non-differentiable rounding operation
5. **ONNX QDQ format** provides an interoperable representation that execution providers can optimize

---

**Next:** [Quantization Techniques — Apply Notebook](./Quantization_Techniques_Apply.ipynb) | [Pruning and Sparsity](../03_Pruning_and_Sparsity/)